In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

class Bookstore:
    def _init_(self): 
        self.inventory = []
        self.sales_data = pd.DataFrame(columns=["date", "title", "quantity sold", "total revenue"])
        self.date_column = "date"

    def add_book(self, title, author, genre, price, quantity):
        title = title.lower()
        author = author.lower()
        genre = genre.lower()
        if price <= 0 or quantity < 0:
            print("Invalid input: Price must be > 0 and Quantity ≥ 0.")
            return
        self.inventory.append({
            "title": title,
            "author": author,
            "genre": genre,
            "price": price,
            "quantity": quantity
        })

    def update_inventory(self, title, quantity):
        title = title.lower()
        for book in self.inventory:
            if book["title"] == title:
                book["quantity"] += quantity
                return
        print("Book not found in inventory.")

    def record_sale(self, title, quantity):
        title = title.lower()
        for book in self.inventory:
            if book["title"] == title:
                if book["quantity"] >= quantity:
                    book["quantity"] -= quantity
                    revenue = book["price"] * quantity
                    new_sale = pd.DataFrame([{
                        self.date_column: pd.Timestamp.today().strftime('%Y-%m-%d'),
                        "title": title,
                        "quantity sold": quantity,
                        "total revenue": revenue
                    }])
                    self.sales_data = pd.concat([self.sales_data, new_sale], ignore_index=True)
                    return
                else:
                    print("Not enough stock.")
                    return
        print("Book not found.")

    def generate_report(self):
        total_books = len(self.inventory)
        if "total revenue" not in self.sales_data.columns:
            print("Error: 'total revenue' column not found in sales data.")
            print("Available columns:", self.sales_data.columns.tolist())
            return

        total_sales = self.sales_data["total revenue"].sum()
        best_seller = self.sales_data.groupby("title")["quantity sold"].sum().idxmax()
        print(f"Total Books in Inventory: {total_books}")
        print(f"Total Revenue: ₹{total_sales:.2f}")
        print(f"Best Selling Book: {best_seller}")

    def load_data(self, inventory_path, sales_path):
        inventory_df = pd.read_csv(inventory_path)
        inventory_df.columns = inventory_df.columns.str.strip().str.lower()
        inventory_df = inventory_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)
        self.inventory = inventory_df.to_dict(orient="records")
        self.sales_data = pd.read_csv(sales_path)
        self.sales_data.columns = self.sales_data.columns.str.strip().str.lower()
        self.sales_data = self.sales_data.applymap(lambda x: x.lower() if isinstance(x, str) else x)

        print("Sales Data Columns:", self.sales_data.columns.tolist())

        for col in self.sales_data.columns:
            if "date" in col:
                self.date_column = col
                break
        else:
            print("⚠ No date column found. Monthly analysis will be skipped.")

    def analyze_sales(self):
        if self.date_column not in self.sales_data.columns:
            print("⚠ Cannot analyze monthly sales: no date column found.")
            return

        self.sales_data[self.date_column] = pd.to_datetime(self.sales_data[self.date_column], errors='coerce')
        monthly_sales = self.sales_data.groupby(self.sales_data[self.date_column].dt.to_period("M")).sum(numeric_only=True)
        print("\nMonthly Sales Summary:\n", monthly_sales)

        genre_sales = pd.DataFrame(self.inventory)
        genre_sales = genre_sales.merge(self.sales_data.groupby("title")["total revenue"].sum(), on="title")
        genre_summary = genre_sales.groupby("genre")["total revenue"].sum()
        print("\nSales by Genre:\n", genre_summary)

    def visualize_data(self):
        author_sales = self.sales_data.groupby("title")["total revenue"].sum().reset_index()
        merged = pd.DataFrame(self.inventory).merge(author_sales, on="title")
        author_group = merged.groupby("author")["total revenue"].sum().sort_values()
        author_group.plot(kind="barh", title="Total Sales by Author", figsize=(10, 6))
        plt.xlabel("Revenue")
        plt.tight_layout()
        plt.show()

        if self.date_column in self.sales_data.columns:
            self.sales_data[self.date_column] = pd.to_datetime(self.sales_data[self.date_column], errors='coerce')
            monthly = self.sales_data.groupby(self.sales_data[self.date_column].dt.to_period("M"))["total revenue"].sum()
            monthly.plot(kind="line", marker="o", title="Monthly Sales Trend", figsize=(10, 6))
            plt.ylabel("Revenue")
            plt.tight_layout()
            plt.show()

        genre_sales = pd.DataFrame(self.inventory).merge(
            self.sales_data.groupby("title")["total revenue"].sum(), on="title"
        )
        genre_group = genre_sales.groupby("genre")["total revenue"].sum()
        genre_group.plot(kind="pie", autopct="%1.1f%%", title="Revenue Share by Genre", figsize=(8, 8))
        plt.ylabel("")
        plt.tight_layout()
        plt.show()

        merged = pd.DataFrame(self.inventory).merge(
            self.sales_data.groupby("title")["quantity sold"].sum(), on="title"
        )
        corr = merged[["price", "quantity sold"]].corr()
        sns.heatmap(corr, annot=True, cmap="coolwarm")
        plt.title("Correlation: Price vs Sales Volume")
        plt.tight_layout()
        plt.show()


if __name__ == "_main_":
    analyzer = Bookstore()

    inventory_path = r"C:\Users\harsh\Desktop\exam\book_sales_data_500.csv"
    sales_path = r"C:\Users\harsh\Desktop\exam\book_sales_transactions_500.csv"

    while True:
        print("\n📚 Bookstore Inventory & Analytics Menu")
        print("1. Load Data")
        print("2. Generate Report")
        print("3. Analyze Sales")
        print("4. Visualize Data")
        print("5. Add Book to Inventory")
        print("6. Record Sale")
        print("7. Exit")

        choice = input("Enter your choice (1–7): ").strip()

        if choice == "1":
            analyzer.load_data(inventory_path, sales_path)
        elif choice == "2":
            analyzer.generate_report()
        elif choice == "3":
            analyzer.analyze_sales()
        elif choice == "4":
            analyzer.visualize_data()
        elif choice == "5":
            title = input("Enter book title: ")
            author = input("Enter author name: ")
            genre = input("Enter genre: ")
            try:
                price = float(input("Enter price: "))
                quantity = int(input("Enter quantity: "))
                analyzer.add_book(title, author, genre, price, quantity)
            except ValueError:
                print("Invalid input. Price must be a number and quantity must be an integer.")
        elif choice == "6":
            title = input("Enter book title for sale: ")
            try:
                quantity = int(input("Enter quantity sold: "))
                analyzer.record_sale(title, quantity)
            except ValueError:
                print("Invalid input. Quantity must be an integer.")
        elif choice == "7":
            print("Exiting... Goodbye!")
            break
        else:
            print("Invalid choice. Please enter a number between 1 and 7.")